# 04 - Linear lap-time model

The baselines in notebook 03 barely beat "no change", and none of them know the lap number. My guess was that fuel burn is the missing piece. This notebook adds a fuel term, gives each compound its own wear slope, and checks. Same setup as before: train on 2022 and 2023, score on 2024, target `delta_s`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from f1lab.features import prepare, split, mae, rmse
from f1lab.models import linear_features, make_linear, raw_coefficients

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
train, val = split(prepare(raw, clean), "dev")
tracks = sorted(train["track"].unique())
FUEL = ["fuel_laps", "fuel_frac"]

def evaluate(name, robust, drop=(), with_tracks=False):
    A = linear_features(train, tracks if with_tracks else None).drop(columns=list(drop))
    B = linear_features(val, tracks if with_tracks else None).drop(columns=list(drop))
    model = make_linear(robust).fit(A, train["delta_s"])
    pred = pd.Series(model.predict(B), index=val.index)
    return {"model": name, "MAE (s)": mae(val["delta_s"], pred), "RMSE (s)": rmse(val["delta_s"], pred)}, model, A, pred

rows = [
    evaluate("OLS, no fuel term", False, drop=FUEL)[0],
    evaluate("OLS, fuel = laps remaining", False, drop=["fuel_frac"])[0],
    evaluate("OLS, fuel = fraction of race left", False, drop=["fuel_laps"])[0],
    evaluate("OLS, both fuel terms", False)[0],
    evaluate("Huber, both fuel terms", True)[0],
    evaluate("Huber, both fuel terms + track effects", True, with_tracks=True)[0],
]
pd.DataFrame(rows).set_index("model").round(3)

,MAE (s),RMSE (s)
model,,
"OLS, no fuel term",0.848,1.336
"OLS, fuel = laps remaining",0.686,1.193
"OLS, fuel = fraction of race left",0.643,1.163
"OLS, both fuel terms",0.638,1.161
"Huber, both fuel terms",0.629,1.161
"Huber, both fuel terms + track effects",0.631,1.166


The guess was right. Adding any fuel term takes the error from about 0.85 s to somewhere between 0.69 and 0.64 s, and the best baseline was 0.855. Using the fraction of the race remaining works better than raw laps remaining, which makes sense because the fuel burned per lap depends on how long the race is. A robust (Huber) fit takes a bit more off (0.629) because incident laps stop dragging it around. A track effect doesn't help (0.631), which fits what I saw earlier: per-track effects are too noisy with two seasons of data.

## What the model learned

In [2]:
metrics, model, A, pred = evaluate("Huber, both fuel terms", True)
coef = raw_coefficients(model, list(A.columns)).round(4)
coef

fuel_laps       -0.0229
fuel_frac        4.3209
offset_SOFT     -0.1851
wear_SOFT        0.0473
offset_MEDIUM   -0.0512
wear_MEDIUM      0.0389
offset_HARD      0.1425
wear_HARD        0.0245
track_temp       0.0011
dtype: float64

In [3]:
# net fuel effect per lap of race remaining, for a short and a long race
for laps in (50, 57, 78):
    print(f"{laps}-lap race: {coef['fuel_frac'] / laps + coef['fuel_laps']:+.3f} s per lap remaining")

50-lap race: +0.064 s per lap remaining
57-lap race: +0.053 s per lap remaining
78-lap race: +0.032 s per lap remaining


What the coefficients say, in seconds:

- Fuel: the two fuel terms work together. In a 57-lap race each extra lap of fuel costs about 0.05 s, and in the 78-lap Monaco race about 0.03 s. A heavier car is slower and the size looks plausible.
- Tyre wear per lap: soft 0.047, medium 0.039, hard 0.025. Softer tyres wear faster, which is the order you'd expect. The regression in notebook 02 is set up differently but gives the same order (0.056, 0.038, 0.029).
- Starting offset on a fresh tyre: soft -0.19 s, medium -0.05, hard +0.14, so about 0.33 s between soft and hard.
- Track temperature: about +0.001 s per degree. Basically nothing, same as in notebook 02.

## Where it is still wrong

In [4]:
err = (val["delta_s"] - pred).abs()
stage = pd.cut(1 - val["laps_remaining"] / val["race_laps"], [0, .25, .5, .75, 1])
print("MAE by race stage:"); print(err.groupby(stage, observed=True).mean().round(3).to_string())
print("\nMAE by compound:"); print(err.groupby(val["compound"]).mean().round(3).to_string())
unseen = val["track"] == "Shanghai"
print(f"\nMAE at Shanghai (no training data): {err[unseen].mean():.3f} | everywhere else: {err[~unseen].mean():.3f}")

MAE by race stage:
(0.0, 0.25]    0.651
(0.25, 0.5]    0.529
(0.5, 0.75]    0.528
(0.75, 1.0]    0.798

MAE by compound:
compound
HARD      0.620
MEDIUM    0.640
SOFT      0.655

MAE at Shanghai (no training data): 0.904 | everywhere else: 0.618


The error is highest in the last quarter of the race (0.80 s against 0.53 to 0.65 earlier) and at Shanghai, a track the model never saw (0.90 against 0.62). I haven't tested why. Possible reasons are late-race traffic and lapped cars, engine and tyre management at the end, wear that isn't linear (a tyre cliff), and wear that differs by track. 0.63 s is still large next to the effects I'm trying to model. Next I'll try a non-linear model that can learn interactions and compare it with this one on the same data.